In [1]:
import pathlib
import sys

_here = pathlib.Path.cwd().resolve()
for _parent in [_here, *_here.parents]:
    if (_parent / "src" / "quant_textbook").exists():
        sys.path.insert(0, str(_parent / "src"))
        break

# 61. Week 41 — Signal research without alpha inflation

## 学習目標


- forecast target、formation lag、candidate count、primary metricを計算前に固定する。
- Treasuryの5公表日先10年yield changeを、zero-changeを残したまま記述的に評価する。
- signal correlationやdirectional accuracyをP&L、causality、tradabilityと呼ばない。


## 前提知識


- B5/B7のchronological validationとpublication horizon
- B3のmultiple testing、estimand、falsification

In [2]:
import time

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

import quant_textbook as qt

pio.renderers.default = "notebook_connected"
RANDOM_SEED = 20260810
NOTEBOOK_ID = 61


def task_rng(task_id, *coordinates):
    entropy = [
        RANDOM_SEED,
        NOTEBOOK_ID,
        int(task_id),
        *(int(coordinate) for coordinate in coordinates),
    ]
    return np.random.default_rng(np.random.SeedSequence(entropy))

In [3]:
treasury = qt.load_treasury_snapshot()
rates = treasury.frame.copy()
forecast = qt.make_treasury_forecast_dataset(rates)
b5_split = qt.chronological_split(len(forecast.regression_target), gap=1)

maturity_years = np.array([0.25, 2.0, 5.0, 10.0, 30.0])
curve_yields = rates.loc[:, qt.DEFAULT_TENORS].to_numpy(dtype=float)
curve_dates = rates["date"].to_numpy(dtype="datetime64[ns]")
curve_changes_bp = np.diff(curve_yields, axis=0) * 100.0
change_dates = curve_dates[1:]

train_end_date = forecast.prediction_dates[b5_split.train.max()]
validation_end_date = forecast.prediction_dates[b5_split.validation.max()]
test_start_date = forecast.prediction_dates[b5_split.test.min()]
train_mask = curve_dates <= train_end_date
validation_mask = (curve_dates > train_end_date) & (curve_dates <= validation_end_date)
test_mask = curve_dates >= test_start_date

assert treasury.quality.accepted
assert train_end_date < validation_end_date < test_start_date
assert np.all(np.isfinite(curve_yields))
assert np.all(np.diff(curve_dates).astype("timedelta64[D]") > np.timedelta64(0, "D"))

print("source:", treasury.metadata.source_name)
print("snapshot:", treasury.metadata.start_date, "to", treasury.metadata.end_date)
print("curve rows / tenors:", curve_yields.shape)
print("B5 train / validation end:", train_end_date, validation_end_date)
print("locked outer-test start:", test_start_date)
print("snapshot sha256:", treasury.metadata.snapshot_sha256)

source: U.S. Treasury Daily Par Yield Curve Rates
snapshot: 2015-01-02 to 2025-12-31
curve rows / tenors: (2750, 5)
B5 train / validation end: 2021-08-11T00:00:00.000000000 2023-10-19T00:00:00.000000000
locked outer-test start: 2023-10-23T00:00:00.000000000
snapshot sha256: 6ddef9605abbf02c6a4526a51f098135b41da1a437915623af672b1c7bcbd295


## 1. Pre-analysis contract

ここでは候補を3個に固定し、5 Treasury publication observations先の10y changeをtargetにする。候補の順位やP&Lは計算後に選ばない。SignalResearchProtocolは主張の境界をデータとして保存する。

$$
y_{t+5}=100(r_{10y,t+5}-r_{10y,t})\quad\text{(basis points)}
$$

In [4]:
protocol = qt.SignalResearchProtocol(
    economic_hypothesis="published curve shape may contain descriptive information about a later 10y change",
    information_timestamp="after the official Treasury publication date",
    target="five-publication-observation 10y par-yield change in bp",
    universe="official 3m/2y/5y/10y/30y par yields",
    holding_period="five Treasury publication observations",
    rebalancing_rule="descriptive signal evaluated at every eligible origin",
    neutralization="none; no asset-pricing portfolio is formed",
    transaction_cost_model="not identified from Treasury par yields",
    primary_metric="correlation and directional accuracy; no P&L claim",
    falsification_test="reverse-time and zero-change baseline remain reported",
    data_source="U.S. Treasury daily par-yield snapshot",
)
horizon = 5
ten_year = curve_yields[:, 3]
target_series = np.zeros(ten_year.size)
target_series[horizon:] = (ten_year[horizon:] - ten_year[:-horizon]) * 100.0
slope_level = (curve_yields[:, 3] - curve_yields[:, 1]) * 100.0
momentum = np.zeros(ten_year.size)
momentum[5:] = (ten_year[5:] - ten_year[:-5]) * 100.0
signals = {
    "zero_change": np.zeros(ten_year.size),
    "10y_minus_2y_level": slope_level,
    "five_observation_momentum": momentum,
}
assert len(signals) == 3
assert protocol.target.startswith("five-publication")

In [5]:
rows = []
for name, signal in signals.items():
    audit = qt.audit_forecast_signal(signal, target_series, formation_lag=horizon, target_unit="bp")
    aligned_target = target_series[horizon:]
    rows.append(
        {
            "candidate": name,
            "observations": audit.observation_count,
            "correlation": audit.correlation,
            "directional_accuracy": audit.directional_accuracy,
            "mean_signed_target_bp": audit.mean_signed_target,
            "pnl_allowed": audit.pnl_interpretation_allowed,
            "falsification_family_size": len(signals),
            "zero_change_rmse_bp": float(np.sqrt(np.mean(aligned_target**2))),
        }
    )
signal_table = pd.DataFrame(rows)
assert signal_table["pnl_allowed"].eq(False).all()
display(signal_table)
fig = go.Figure()
fig.add_bar(x=signal_table["candidate"], y=signal_table["correlation"], name="correlation")
fig.add_bar(x=signal_table["candidate"], y=signal_table["directional_accuracy"], name="directional accuracy")
fig.update_layout(
    title="Pre-registered descriptive signal diagnostics",
    xaxis_title="Candidate (none selected after seeing the result)",
    yaxis_title="Diagnostic value",
    barmode="group",
    template="plotly_white",
)
fig.show()

,candidate,observations,correlation,directional_accuracy,mean_signed_target_bp,pnl_allowed,falsification_family_size,zero_change_rmse_bp
0,zero_change,2745,0.000000,0.044809,0.000000,False,3,11.490695
1,10y_minus_2y_level,2745,-0.025446,0.460474,0.028415,False,3,11.490695
2,five_observation_momentum,2745,-0.013399,0.445902,-0.105282,False,3,11.490695


## 2. 失敗モード

- signalをtargetと同じ時点へずらす。
- 3候補を試して最良だけをprimaryへ書き換える。
- mean_signed_target_bpをstrategy returnと呼ぶ。
- Treasury par yieldからexecution costやcausal effectを作る。

## 3. 段階別演習

### 基礎

1. target_seriesのindexを図で確認し、5観測先の情報集合を説明せよ。

### 標準

2. reverse-time falsificationを追加し、候補数を増やしたときのselection riskを書け。

### 研究

3. Fama–MacBethをcross-sectional exposureへ拡張する場合の必要なasset return dataとclustered inferenceを定義せよ。

## 4. Exit Criteria

- [ ] signal計算前に候補数とprimary metricを固定した
- [ ] formation lagがtargetを未来へ置いた
- [ ] zero-change baselineを残した
- [ ] descriptive diagnosticsとP&L claimを分けた
- [ ] falsificationとmultiple-testingの対象を記録した

## 5. 出典

- [U.S. Treasury Yield Curve Methodology](https://home.treasury.gov/policy-issues/financing-the-government/interest-rate-statistics/treasury-yield-curve-methodology)
- [U.S. Treasury Daily Treasury Par Yield Curve Rates](https://home.treasury.gov/resource-center/data-chart-center/interest-rates/TextView?type=daily_treasury_yield_curve)

- [Fama and MacBeth (1973), Risk, Return, and Equilibrium](https://www.jstor.org/stable/1831028)
- [Hansen (1982), Large Sample Properties of GMM Estimators](https://doi.org/10.2307/1912775)
- [Duffie and Kan (1996), A Yield-Factor Model of Interest Rates](https://doi.org/10.1016/0304-405X(95)00881-6)
- [Boyd and Vandenberghe, Convex Optimization](https://web.stanford.edu/~boyd/cvxbook/bv_cvxbook.pdf)